In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
BRANCH = "main"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("Current repo:", os.getcwd())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes peft transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train.py wad_dataset.py qformer_bridge.py scripts/test_infer.py scripts/prepare_qformer.py scripts/smoke_qformer_bridge.py


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config internvl_config.yaml


In [ ]:
!python scripts/smoke_qformer_bridge.py --config internvl_config.yaml


In [ ]:
import subprocess

TRAIN_CHECKPOINT = ""  # local folder or HF repo id; leave empty to train from default setup
TRAIN_START_EPOCH = None  # set when resuming from HF repo or custom checkpoint naming
TRAIN_START_STEP = None   # set when resuming from HF repo or custom checkpoint naming
cmd = ["python", "train.py"]
if TRAIN_CHECKPOINT:
    cmd += ["--checkpoint", TRAIN_CHECKPOINT]
if TRAIN_START_EPOCH is not None:
    cmd += ["--start_epoch", str(TRAIN_START_EPOCH)]
if TRAIN_START_STEP is not None:
    cmd += ["--start_step", str(TRAIN_START_STEP)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import subprocess
from pathlib import Path

CHECKPOINT_DIR = ""  # optional: fill manually with outputs/.../epoch_x or epoch_x_step_y
if not CHECKPOINT_DIR:
    candidates = sorted(Path("outputs/internvl3_2b").glob("*/*"), key=lambda p: p.stat().st_mtime)
    candidates = [p for p in candidates if (p / "adapter_config.json").exists()]
    CHECKPOINT_DIR = str(candidates[-1]) if candidates else ""

if CHECKPOINT_DIR:
    print("Evaluating checkpoint:", CHECKPOINT_DIR)
    subprocess.run([
        "python", "scripts/test_infer.py",
        "--config", "internvl_config.yaml",
        "--checkpoint", CHECKPOINT_DIR,
        "--split", "test_QA",
        "--output_file", "results/qformer_eval_test_QA.json",
    ], check=True)
else:
    print("No checkpoint found. Train first, then rerun this cell.")


In [ ]:
import subprocess
subprocess.run(["python", "scripts/visualize_training.py", "--save", "training_plot.png"], check=True)

from IPython.display import Image, display
display(Image("training_plot.png"))


In [ ]:
import json
from pathlib import Path

base = Path("outputs/internvl3_2b")
candidates = sorted(base.glob("*/metrics.json"), key=lambda p: p.stat().st_mtime, reverse=True)
METRICS_PATH = str(candidates[0])

with open(METRICS_PATH) as f:
    m = json.load(f)

print("=== EPOCH SUMMARY ===")
for ep in m["epoch_summary"]:
    print(f"  Epoch {ep['epoch']}: avg_train_loss = {ep['avg_train_loss']:.6f}")

print("\n=== VALIDATION LOSS ===")
for v in m["val_loss"]:
    print(f"  Epoch {v['epoch']} | Step {v['step']:>6}: {v['loss']:.6f}")
if m["val_loss"]:
    best = min(m["val_loss"], key=lambda x: x["loss"])
    print(f"\n  Best: {best['loss']:.6f} @ step {best['step']}")
